# Import libraries

In [0]:
%restart_python

In [0]:
import pandas as pd
from collections import Counter

In [0]:
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Spark session

In [0]:
# create session

spark = SparkSession.builder.appName("ALS").getOrCreate()

# Read data

In [0]:
# https://grouplens.org/datasets/movielens/
# select the ml-latest-small.zip dataset

In [0]:
df_rating = pd.read_csv('./ALS_data/ratings.csv')

In [0]:
df_rating['fecha'] = pd.to_datetime(df_rating['timestamp'], unit='s')

Rating explicito

![](https://imgs.search.brave.com/HA01A6Cnbdwd2soNPh-ETOrdfOZy1AY-3I7k1K5HUD4/rs:fit:860:0:0:0/g:ce/aHR0cHM6Ly9zdGF0/aWMudmVjdGVlenku/Y29tL3N5c3RlbS9y/ZXNvdXJjZXMvdGh1/bWJuYWlscy8wNzUv/Mjg4LzgzNi9zbWFs/bC9zdGFyLXJhdGlu/Zy1zY2FsZS1mcm9t/LWxvdy10by1oaWdo/LXdpdGgtcGVyY2Vu/dGFnZS1sZXZlbHMt/Zm9yLWZlZWRiYWNr/LWFuZC1ldmFsdWF0/aW9uLXZlY3Rvci5q/cGc)

In [0]:
df_rating

,userId,movieId,rating,timestamp,fecha
0,1,1,4.0,964982703,2000-07-30 18:45:03
1,1,3,4.0,964981247,2000-07-30 18:20:47
2,1,6,4.0,964982224,2000-07-30 18:37:04
3,1,47,5.0,964983815,2000-07-30 19:03:35
4,1,50,5.0,964982931,2000-07-30 18:48:51
...,...,...,...,...,...
100831,610,166534,4.0,1493848402,2017-05-03 21:53:22
100832,610,168248,5.0,1493850091,2017-05-03 22:21:31
100833,610,168250,5.0,1494273047,2017-05-08 19:50:47
100834,610,168252,5.0,1493846352,2017-05-03 21:19:12


In [0]:
df_movies = pd.read_csv('./ALS_data/movies.csv')

In [0]:
df_movies

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [0]:
df_movies[df_movies.title.str.contains('Home Alone')]

,movieId,title,genres
504,586,Home Alone (1990),Children|Comedy
1285,1707,Home Alone 3 (1997),Children|Comedy
2224,2953,Home Alone 2: Lost in New York (1992),Children|Comedy
8625,118880,"Girl Walks Home Alone at Night, A (2014)",Horror|Romance|Thriller


In [0]:
df_join_ = df_rating.merge(df_movies, on='movieId', how='left')

In [0]:
df_join_

,userId,movieId,rating,timestamp,fecha,title,genres
0,1,1,4.0,964982703,2000-07-30 18:45:03,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,2000-07-30 18:20:47,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,2000-07-30 18:37:04,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,2000-07-30 19:03:35,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,2000-07-30 18:48:51,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
...,...,...,...,...,...,...,...
100831,610,166534,4.0,1493848402,2017-05-03 21:53:22,Split (2017),Drama|Horror|Thriller
100832,610,168248,5.0,1493850091,2017-05-03 22:21:31,John Wick: Chapter Two (2017),Action|Crime|Thriller
100833,610,168250,5.0,1494273047,2017-05-08 19:50:47,Get Out (2017),Horror
100834,610,168252,5.0,1493846352,2017-05-03 21:19:12,Logan (2017),Action|Sci-Fi


In [0]:
df_join_[df_join_.title.str.contains('Home Alone')].groupby('title').agg({'rating':'mean','userId':'nunique','timestamp':'count'})

,rating,userId,timestamp
title,,,
"Girl Walks Home Alone at Night, A (2014)",3.500000,2,2
Home Alone (1990),2.995690,116,116
Home Alone 2: Lost in New York (1992),2.516129,31,31
Home Alone 3 (1997),1.875000,8,8


In [0]:
df_rating

,userId,movieId,rating,timestamp,fecha
0,1,1,4.0,964982703,2000-07-30 18:45:03
1,1,3,4.0,964981247,2000-07-30 18:20:47
2,1,6,4.0,964982224,2000-07-30 18:37:04
3,1,47,5.0,964983815,2000-07-30 19:03:35
4,1,50,5.0,964982931,2000-07-30 18:48:51
...,...,...,...,...,...
100831,610,166534,4.0,1493848402,2017-05-03 21:53:22
100832,610,168248,5.0,1493850091,2017-05-03 22:21:31
100833,610,168250,5.0,1494273047,2017-05-08 19:50:47
100834,610,168252,5.0,1493846352,2017-05-03 21:19:12


In [0]:
df_rating.userId.nunique()

610

In [0]:
df_rating.movieId.nunique()

9724

In [0]:
df_movies

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [0]:
df_movies.movieId.nunique()

9742

# Spark dataframes

In [0]:
# create a spark dataframe

df_spark_rating = spark.createDataFrame(df_rating)

In [0]:
df_spark_rating.show(5)

+------+-------+------+---------+-------------------+
|userId|movieId|rating|timestamp|              fecha|
+------+-------+------+---------+-------------------+
|     1|      1|   4.0|964982703|2000-07-30 18:45:03|
|     1|      3|   4.0|964981247|2000-07-30 18:20:47|
|     1|      6|   4.0|964982224|2000-07-30 18:37:04|
|     1|     47|   5.0|964983815|2000-07-30 19:03:35|
|     1|     50|   5.0|964982931|2000-07-30 18:48:51|
+------+-------+------+---------+-------------------+
only showing top 5 rows


In [0]:
df_spark_rating.printSchema()

root
 |-- userId: long (nullable = true)
 |-- movieId: long (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- fecha: timestamp (nullable = true)



# Split data set

![](https://miro.medium.com/1*4G__SV580CxFj78o9yUXuQ.png)

In [0]:
# split data set

(df_training, df_test) = df_spark_rating.randomSplit([0.8, 0.2], seed=42)

In [0]:
df_training.show(5)

+------+-------+------+---------+-------------------+
|userId|movieId|rating|timestamp|              fecha|
+------+-------+------+---------+-------------------+
|     1|      1|   4.0|964982703|2000-07-30 18:45:03|
|     1|      3|   4.0|964981247|2000-07-30 18:20:47|
|     1|      6|   4.0|964982224|2000-07-30 18:37:04|
|     1|     47|   5.0|964983815|2000-07-30 19:03:35|
|     1|     50|   5.0|964982931|2000-07-30 18:48:51|
+------+-------+------+---------+-------------------+
only showing top 5 rows


In [0]:
df_test.show(5)

+------+-------+------+---------+-------------------+
|userId|movieId|rating|timestamp|              fecha|
+------+-------+------+---------+-------------------+
|     1|    151|   5.0|964984041|2000-07-30 19:07:21|
|     1|    216|   5.0|964981208|2000-07-30 18:20:08|
|     1|    231|   5.0|964981179|2000-07-30 18:19:39|
|     1|    362|   5.0|964982588|2000-07-30 18:43:08|
|     1|    423|   3.0|964982363|2000-07-30 18:39:23|
+------+-------+------+---------+-------------------+
only showing top 5 rows


In [0]:
df_training.count(), df_test.count(), df_spark_rating.count()

(80960, 19876, 100836)

In [0]:
# check obs

df_training.count() + df_test.count()

100836

# Train model

In [0]:
# define ALS parameters

als = ALS(
    maxIter=10,
    regParam=0.1,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=42
)

In [0]:
# train the model

model = als.fit(df_training)

In [0]:
# calcualte predictions

predictions = model.transform(df_test)

In [0]:
predictions.show(5)

+------+-------+------+----------+-------------------+----------+
|userId|movieId|rating| timestamp|              fecha|prediction|
+------+-------+------+----------+-------------------+----------+
|    29|    914|   4.0|1362016608|2013-02-28 01:56:48| 4.0180883|
|    29|    953|   5.0|1307906372|2011-06-12 19:19:32| 4.1815023|
|    29|   1204|   4.5|1308084015|2011-06-14 20:40:15| 4.3076787|
|    29|   1408|   5.0|1405816272|2014-07-20 00:31:12|  4.007029|
|    29|   1772|   1.5|1307906072|2011-06-12 19:14:32| 2.9508288|
+------+-------+------+----------+-------------------+----------+
only showing top 5 rows


# Check performance

![](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQ0rU4BqXX7j61kBo8NXhS9nVVb9Qfn_c5QTA&s)

In [0]:
# check model performance

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

In [0]:
# calcualte RMSE

rmse = evaluator.evaluate(predictions)

In [0]:
print(f" RMSE (Root Mean Squared Error): {rmse:.4f}")

 RMSE (Root Mean Squared Error): 0.8761


![](https://miro.medium.com/1*5fnmYVHLTC8mGxybHm4XkA.png)

In [0]:
# check model performance

evaluator_r2 = RegressionEvaluator(
    metricName="r2",
    labelCol="rating",
    predictionCol="prediction"
)

# calcualte MAPE

r2_ = evaluator_r2.evaluate(predictions)

print(f" r2 : {r2_:.4f}")

 r2 : 0.2779


# Recommendations

In [0]:
# cross user vs movie

user_movie = df_spark_rating.select("userId").distinct().crossJoin(
    df_spark_rating.select("movieId").distinct()
)

# calculate all predictions

predictions = model.transform(user_movie)

# select top 3 rating scored

window = Window.partitionBy("userId").orderBy(col("prediction").desc()) # similar to SQL command ROW_NUMBER() OVER(PARTITION BY userId ORDER BY prediction DESC)

df_all_recos = predictions.withColumn("rank", F.row_number().over(window)) \
                      .filter(col("rank") <= 3)


In [0]:
df_all_recos.show(10)

+------+-------+----------+----+
|userId|movieId|prediction|rank|
+------+-------+----------+----+
|     1|  26171| 5.8742876|   1|
|     1| 177593|  5.797969|   2|
|     1| 170355| 5.6960335|   3|
|     2| 170355|  4.909435|   1|
|     2|   3379|  4.909435|   2|
|     2|  68945|  4.909435|   3|
|     3|   5746| 4.8837748|   1|
|     3|   6835| 4.8837748|   2|
|     3|  70946| 4.7963963|   3|
|     4|   2524|  5.753259|   1|
+------+-------+----------+----+
only showing top 10 rows


In [0]:
df_pd_recs = df_all_recos.toPandas()

In [0]:
df_pd_recs

,userId,movieId,prediction,rank
0,1,26171,5.874288,1
1,1,177593,5.797969,2
2,1,170355,5.696033,3
3,2,170355,4.909435,1
4,2,3379,4.909435,2
...,...,...,...,...
1825,609,184245,4.229933,2
1826,609,134796,4.229933,3
1827,610,170355,5.486994,1
1828,610,3379,5.486994,2


In [0]:
def retrieval_recommendations(p_user_id,last_movies = 10):

    # retrieval favorites movies watched

    df_top_10 = df_rating[df_rating['userId'] == p_user_id].sort_values(by=['rating','fecha'], ascending=[False,False]).head(last_movies).merge(df_movies, on = 'movieId', how = 'left')

    df_top_10['genres'] = df_top_10['genres'].apply(lambda x: x + '|')

    list_genders = df_top_10.genres.sum().split('|')[:-1]

    df_profile = pd.DataFrame(Counter(list_genders).most_common(20), columns = ['genres', 'count'])
    df_profile['pct'] = df_profile['count']/len(list_genders)

    # show profile
    
    display(df_profile)

    list_movies_watched = df_rating[df_rating['userId'] == p_user_id].movieId.values.tolist()

    df_reco_ = df_pd_recs[(df_pd_recs.userId == p_user_id) & (~df_pd_recs.movieId.isin(list_movies_watched))].merge(df_movies, on = 'movieId', how = 'left')

    return df_reco_

In [0]:
retrieval_recommendations(5)

genres,count,pct
Drama,8,0.25
Crime,3,0.09375
Animation,3,0.09375
Children,3,0.09375
Fantasy,3,0.09375
Musical,3,0.09375
Comedy,2,0.0625
Romance,2,0.0625
War,1,0.03125
IMAX,1,0.03125


,userId,movieId,prediction,rank,title,genres
0,5,177593,5.079135,1,"Three Billboards Outside Ebbing, Missouri (2017)",Crime|Drama
1,5,132333,5.060704,2,Seve (2014),Documentary|Drama
2,5,8542,5.060704,3,"Day at the Races, A (1937)",Comedy|Musical


In [0]:
retrieval_recommendations(90)

genres,count,pct
Drama,7,0.35
Comedy,5,0.25
Romance,3,0.15
Crime,2,0.1
Thriller,1,0.05
Musical,1,0.05
Documentary,1,0.05


,userId,movieId,prediction,rank,title,genres
0,90,3925,5.576566,1,Stranger Than Paradise (1984),Comedy|Drama
1,90,6818,5.575766,2,Come and See (Idi i smotri) (1985),Drama|War
2,90,8477,5.560734,3,"Jetée, La (1962)",Romance|Sci-Fi


In [0]:
retrieval_recommendations(200)

genres,count,pct
Comedy,9,0.42857142857142855
Drama,2,0.09523809523809523
Crime,2,0.09523809523809523
Action,2,0.09523809523809523
Thriller,2,0.09523809523809523
Children,1,0.047619047619047616
Adventure,1,0.047619047619047616
Romance,1,0.047619047619047616
Mystery,1,0.047619047619047616


,userId,movieId,prediction,rank,title,genres
0,200,945,5.217923,1,Top Hat (1935),Comedy|Musical|Romance
1,200,74226,5.139302,2,"Dream of Light (a.k.a. Quince Tree Sun, The) (...",Documentary|Drama
2,200,26928,5.139302,3,"Summer's Tale, A (Conte d'été) (1996)",Comedy|Drama|Romance


![image_1777494948236.png](./image_1777494948236.png "image_1777494948236.png")